In [1]:
import yfinance as yf
import pandas as pd
import sqlite3
from datetime import datetime

In [2]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'META', 'AMZN', 'NVDA', 'AMD',
           'INTC', 'CRM', 'ORCL', 'ADBE', 'CSCO', 'IBM', 'NOW', 'NFLX']

BENCHMARKS = ['SPY', 'XLK']

START_DATE = '2019-01-01'
END_DATE = '2025-01-01'

ALL_SYMBOLS = TICKERS + BENCHMARKS

In [3]:
data = yf.download(
    ALL_SYMBOLS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True
)

print(f"Shape: {data.shape[0]} rows (trading days) x {data.shape[1]} columns")

[*********************100%***********************]  17 of 17 completed

Shape: 1510 rows (trading days) x 85 columns


In [4]:
data.head()

Price           Close                                                \
Ticker           AAPL        ADBE        AMD       AMZN         CRM   
2019-01-02  37.469208  224.570007  18.830000  76.956497  133.588715   
2019-01-03  33.737000  215.699997  17.049999  75.014000  128.513199   
2019-01-04  35.177216  226.190002  19.000000  78.769501  135.963837   
2019-01-07  35.098900  229.259995  20.570000  81.475502  140.162201   
2019-01-08  35.767994  232.679993  20.750000  82.829002  143.611572   

Price                                                               ...  \
Ticker           CSCO      GOOGL        IBM       INTC        META  ...   
2019-01-02  34.472191  52.301727  80.143333  40.518040  134.623550  ...   
2019-01-03  33.218513  50.853195  78.543396  38.289040  130.714218  ...   
2019-01-04  34.714836  53.461643  81.611122  40.638535  136.875885  ...   
2019-01-07  34.949394  53.355026  82.188477  40.827866  136.975082  ...   
2019-01-08  35.232494  53.823650  83.357132  41.086063  141.420212  ...   

Price        Volume                                                     \
Ticker          IBM      INTC      META      MSFT       NFLX       NOW   
2019-01-02  4434935  18774600  28146200  35329300  116795000  12807500   
2019-01-03  4546648  32267300  22717900  42579100  149696000  12344500   
2019-01-04  4683779  35447300  29002100  44060600  193301000  10978500   
2019-01-07  3923755  22736800  20089300  35656100  186201000   9124500   
2019-01-08  4982726  22749200  26263800  31514400  153592000   8046000   

Price                                                 
Ticker           NVDA      ORCL        SPY       XLK  
2019-01-02  508752000  14320400  126925200  30885800  
2019-01-03  705552000  19868700  144140700  49893400  
2019-01-04  585620000  20984000  142628800  41535600  
2019-01-07  709160000  17967900  103139100  23817200  
2019-01-08  786016000  16255700  102512600  26005200  

[5 rows x 85 columns]

In [5]:
data['Close'].head()

Ticker,AAPL,ADBE,AMD,AMZN,CRM,CSCO,GOOGL,IBM,INTC,META,MSFT,NFLX,NOW,NVDA,ORCL,SPY,XLK
2019-01-02,37.469208,224.570007,18.830000,76.956497,133.588715,34.472191,52.301727,80.143333,40.518040,134.623550,94.193123,26.766001,35.664001,3.376983,40.505608,224.382553,29.032202
2019-01-03,33.737000,215.699997,17.049999,75.014000,128.513199,33.218513,50.853195,78.543396,38.289040,130.714218,90.727966,27.120001,33.824001,3.172956,40.111473,219.028137,27.567020
2019-01-04,35.177216,226.190002,19.000000,78.769501,135.963837,34.714836,53.461643,81.611122,40.638535,136.875885,94.947655,29.757000,35.846001,3.376239,41.840267,226.364670,28.788786
2019-01-07,35.098900,229.259995,20.570000,81.475502,140.162201,34.949394,53.355026,82.188477,40.827866,136.975082,95.068726,31.534000,37.334000,3.554981,42.503117,228.149399,29.046244
2019-01-08,35.767994,232.679993,20.750000,82.829002,143.611572,35.232494,53.823650,83.357132,41.086063,141.420212,95.758057,32.027000,37.622002,3.466477,42.888283,230.293045,29.289663


In [6]:
print("Missing Close prices per ticker:")
print(data['Close'].isna().sum())

Missing Close prices per ticker:
Ticker
AAPL     0
ADBE     0
AMD      0
AMZN     0
CRM      0
CSCO     0
GOOGL    0
IBM      0
INTC     0
META     0
MSFT     0
NFLX     0
NOW      0
NVDA     0
ORCL     0
SPY      0
XLK      0
dtype: int64


In [7]:
long_data = data.stack(level='Ticker', future_stack=True).reset_index()

long_data.columns = [c.lower() for c in long_data.columns]
long_data = long_data.rename(columns={'level_0': 'date'})

print(f"Shape: {long_data.shape}")
long_data.head(10)

Shape: (25670, 7)


,date,ticker,close,high,low,open,volume
0,2019-01-02,AAPL,37.469208,37.689868,36.593692,36.750289,148158800
1,2019-01-02,ADBE,224.570007,226.169998,219.000000,219.910004,2784100
2,2019-01-02,AMD,18.830000,19.000000,17.980000,18.010000,87148700
3,2019-01-02,AMZN,76.956497,77.667999,73.046501,73.260002,159662000
4,2019-01-02,CRM,133.588715,134.850193,131.124887,131.469814,4783900
5,2019-01-02,CSCO,34.472191,34.672844,33.878257,33.934439,23833500
6,2019-01-02,GOOGL,52.301727,52.604723,50.843776,50.938990,31868000
7,2019-01-02,IBM,80.143333,80.678967,77.694725,77.917323,4434935
8,2019-01-02,INTC,40.518040,40.853681,39.390625,39.554142,18774600
9,2019-01-02,META,134.623550,136.439303,127.558994,127.985653,28146200


In [8]:
import sqlite3

DB_PATH = "../data/market.db"

conn = sqlite3.connect(DB_PATH)

long_data.to_sql('prices', conn, if_exists='replace', index=False)

result = conn.execute("SELECT COUNT(*) FROM prices").fetchone()
print(f"Saved {result[0]} rows to the 'prices' table in {DB_PATH}")

conn.close()
print("Database connection closed.")

Saved 25670 rows to the 'prices' table in ../data/market.db
Database connection closed.


In [9]:
import time

earnings_frames = []

for ticker in TICKERS:
    try:
        t = yf.Ticker(ticker)
        ed = t.get_earnings_dates(limit=60)
        if ed is not None and len(ed) > 0:
            ed = ed.reset_index()
            ed['ticker'] = ticker
            earnings_frames.append(ed)
            print(f"{ticker}: {len(ed)} earnings dates")
        else:
            print(f"{ticker}: no earnings dates returned")
    except Exception as e:
        print(f"{ticker}: ERROR - {e}")
    time.sleep(1)

print(f"\nCollected earnings data for {len(earnings_frames)} companies")

AAPL: 100 earnings dates
MSFT: 100 earnings dates
GOOGL: 88 earnings dates
META: 57 earnings dates
AMZN: 100 earnings dates
NVDA: 100 earnings dates
AMD: 100 earnings dates
INTC: 100 earnings dates
CRM: 89 earnings dates
ORCL: 100 earnings dates
ADBE: 100 earnings dates
CSCO: 100 earnings dates
IBM: 100 earnings dates
NOW: 57 earnings dates
NFLX: 97 earnings dates

Collected earnings data for 15 companies


In [10]:
earnings_df = pd.concat(earnings_frames, ignore_index=True)

earnings_df.columns = [c.lower().replace(' ', '_') for c in earnings_df.columns]

print(f"Total rows: {len(earnings_df)}")
print(f"Columns: {earnings_df.columns.tolist()}")
earnings_df.head(10)

Total rows: 1388
Columns: ['earnings_date', 'eps_estimate', 'reported_eps', 'surprise(%)', 'ticker']


,earnings_date,eps_estimate,reported_eps,surprise(%),ticker
0,2026-07-30 16:00:00-04:00,1.90,NaN,NaN,AAPL
1,2026-04-30 16:00:00-04:00,1.94,2.01,3.46,AAPL
2,2026-01-29 16:00:00-05:00,2.67,2.84,6.25,AAPL
3,2025-10-30 16:00:00-04:00,1.77,1.85,4.52,AAPL
4,2025-07-31 16:00:00-04:00,1.43,1.57,9.48,AAPL
5,2025-05-01 16:00:00-04:00,1.63,1.65,1.50,AAPL
6,2025-01-30 16:00:00-05:00,2.35,2.40,2.26,AAPL
7,2024-10-31 16:00:00-04:00,1.60,1.64,2.28,AAPL
8,2024-08-01 16:00:00-04:00,1.34,1.40,4.30,AAPL
9,2024-05-02 16:00:00-04:00,1.51,1.53,1.46,AAPL


In [11]:
earnings_df = earnings_df.rename(columns={'surprise(%)': 'eps_surprise_pct'})
earnings_df['earnings_date'] = pd.to_datetime(earnings_df['earnings_date'], utc=True).dt.tz_localize(None)
earnings_df = earnings_df[earnings_df['earnings_date'] >= '2019-01-01']
print(f"Rows after filtering to 2019+: {len(earnings_df)}")

Rows after filtering to 2019+: 463


In [12]:
conn = sqlite3.connect("../data/market.db")
earnings_df.to_sql('earnings', conn, if_exists='replace', index=False)
count = conn.execute("SELECT COUNT(*) FROM earnings").fetchone()[0]
print(f"Saved {count} rows to 'earnings' table")
conn.close()

Saved 463 rows to 'earnings' table


In [19]:
# Strip time component so dates match the prices table
earnings_df['earnings_date'] = pd.to_datetime(earnings_df['earnings_date']).dt.date
earnings_df['earnings_date'] = pd.to_datetime(earnings_df['earnings_date'])

# Rerun the return calculation
rows = []
for _, row in earnings_df.iterrows():
    if pd.isna(row['reported_eps']):
        continue
    date = row['earnings_date']
    ticker = row['ticker']
    r1  = get_return(prices, ticker, date, 1)
    r30 = get_return(prices, ticker, date, 30)
    r90 = get_return(prices, ticker, date, 90)
    spy1  = get_return(prices, 'SPY', date, 1)
    spy30 = get_return(prices, 'SPY', date, 30)
    spy90 = get_return(prices, 'SPY', date, 90)
    rows.append({
        'ticker': ticker,
        'earnings_date': date,
        'return_1d': r1,
        'return_30d': r30,
        'return_90d': r90,
        'abnormal_1d':  r1  - spy1  if r1  and spy1  else None,
        'abnormal_30d': r30 - spy30 if r30 and spy30 else None,
        'abnormal_90d': r90 - spy90 if r90 and spy90 else None,
    })

returns_df = pd.DataFrame(rows)
print(f"Computed returns for {len(returns_df)} earnings events")
returns_df.head(10)

Computed returns for 448 earnings events


,ticker,earnings_date,return_1d,return_30d,return_90d,abnormal_1d,abnormal_30d,abnormal_90d
0,AAPL,2026-04-30,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,2025-10-30,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL,2025-07-31,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL,2025-05-01,NaN,NaN,NaN,NaN,NaN,NaN
5,AAPL,2025-01-30,NaN,NaN,NaN,NaN,NaN,NaN
6,AAPL,2024-10-31,-0.013280,0.099566,0.109714,-0.017500,0.037013,0.075575
7,AAPL,2024-08-01,0.006869,0.020139,0.132567,0.025488,-0.014851,0.015581
8,AAPL,2024-05-02,0.059816,0.229719,0.290066,0.047421,0.154971,0.188728
9,AAPL,2024-02-01,-0.005405,-0.075028,0.111504,-0.015932,-0.120437,0.010485


In [22]:
conn = sqlite3.connect("../data/market.db")
returns_df.to_sql('returns', conn, if_exists='replace', index=False)
print(f"Saved {len(returns_df)} rows to 'returns' table")
conn.close()

Saved 448 rows to 'returns' table
